# 06 · Implantação

**CRISP-DM fase 6 — implantação** · Meridiano Imobiliária

---

Atribui um segmento a imóveis novos em três passos:
1. **modelo:** lê o modelo vencedor do [`04-0-modelagem`](04-0-modelagem.ipynb);
2. **entrada:** o imóvel novo em unidades de negócio (pés², nota, bairro…) passa pelo mesmo pré-processamento do [`03-preparacao`](03-preparacao.ipynb);
3. **saída:** o segmento do imóvel, com o perfil desse segmento na carteira.

In [1]:
import json
import warnings
warnings.filterwarnings('ignore')  # silencia avisos de versão do scikit-learn

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.spatial.distance import cdist
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

DADOS = Path('../data/processed')
MODELOS = Path('../models')

## 1 · Modelo treinado

In [3]:
selecionado = json.loads((MODELOS / 'selecionado.json').read_text())  # vencedor e parâmetros, salvos pelo 04-0

try:
    modelo = joblib.load(MODELOS / selecionado['arquivo'])              # modelo treinado
except Exception as erro:                                                # k-medianas/k-medoides só carregam onde a classe foi definida
    modelo = None
    print(f'Modelo não carregado ({type(erro).__name__}): a atribuição usa o centroide mais próximo')

classifica_novos = hasattr(modelo, 'predict')                            # ward e dbscan não classificam imóveis novos
display(Markdown(
    f"Modelo: **{selecionado['modelo']}** · `{selecionado['params']}` · {selecionado['grupos']} segmentos · "
    f"atribuição por **{'predict do modelo' if classifica_novos else 'centroide mais próximo'}**"
))

Modelo: **mini batch k-means** · `{'batch_size': 256, 'n_clusters': 4}` · 4 segmentos · atribuição por **predict do modelo**

## 2 · Pré-processamento do 03

O 03 salvou a base transformada, mas não o pré-processador. Aqui ele é refeito com as mesmas regras e ajustado na mesma base curada, então as escalas são as mesmas. A célula confere que o resultado bate com a saída do 03 antes de seguir.

`preco_venda` e `subclasse` ficam fora, como no 04-0: o preço de um imóvel novo nem sempre é conhecido.

In [4]:
bruta = pd.read_csv(DADOS / 'ames_housing_tratado.csv', sep=';')              # base curada: referência de escalas e categorias
dicionario = pd.read_csv(DADOS / 'dicionario.csv', sep=';')                     # tipo de cada variável
escala = pd.read_csv(DADOS / 'escala.csv', sep=';', dtype={'nivel': str}).sort_values(['coluna', 'ordem'])
escalas = escala.groupby('coluna')['nivel'].apply(list).to_dict()             # níveis ordenados de cada ordinal

FORA = ['ordem', 'pid', 'preco_venda', 'subclasse']                             # o que não entra no modelo
bruta_modelo = bruta.drop(columns=['ordem', 'pid'], errors='ignore')            # colunas que o pré-processador recebe


def colunas_do_tipo(*tipos):
    colunas = dicionario.loc[dicionario['Tipo'].isin(tipos), 'variavel']
    return [c for c in colunas if c in bruta.columns and c not in FORA]


derivadas = ['area_varandas', 'banheiros_total', 'idade_imovel', 'anos_desde_reforma',             # criadas no 02
             'tem_porao', 'tem_garagem', 'tem_lareira', 'tem_segundo_pavimento', 'foi_reformada']
continuas = list(dict.fromkeys(colunas_do_tipo('Contínua', 'Discreta') + [c for c in derivadas if c in bruta]))
ordinais = colunas_do_tipo('Ordinal')
nominais = colunas_do_tipo('Nominal')

niveis = {}
for col in ordinais:                                                            # mesma regra de codificação do 03
    escala_col = pd.to_numeric(pd.Series(escalas[col])).tolist() if pd.api.types.is_numeric_dtype(bruta[col]) else escalas[col]
    ausencia = [v for v in bruta[col].unique() if isinstance(v, str) and v.startswith('Sem ') and v not in escala_col]
    niveis[col] = ausencia + escala_col                                         # "Sem porão", "Sem garagem"… entram como o menor nível


def codificar(dados):
    # Troca cada nível ordinal pela sua posição na escala (0, 1, 2…).
    dados = dados.copy()
    for col in ordinais:
        codigos = pd.Categorical(dados[col], categories=niveis[col], ordered=True).codes
        if (codigos < 0).any():
            raise ValueError(f'{col}: valor fora da escala. Use um destes: {niveis[col]}')
        dados[col] = codigos
    return dados


preprocessador = ColumnTransformer([
    ('continuous', MinMaxScaler(), continuas),
    ('ordinal', MinMaxScaler(), ordinais),
    ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominais),
], verbose_feature_names_out=False).set_output(transform='pandas').fit(codificar(bruta_modelo))

X_base = preprocessador.transform(codificar(bruta_modelo))                                       # carteira no espaço do modelo
saida_03 = pd.read_csv(DADOS / 'ames_housing_preprocessed.csv', sep=';').drop(columns=FORA, errors='ignore')
assert list(X_base.columns) == list(saida_03.columns) and np.allclose(X_base, saida_03), 'o pré-processamento não reproduz o 03'
print(f'Pré-processamento confere com o 03: {X_base.shape[1]} features')

Pré-processamento confere com o 03: 230 features


In [5]:
segmentos = pd.read_csv(DADOS / 'segmentos.csv', sep=';')['segmento'].to_numpy()  # rótulos da carteira (04-0)
validos = segmentos >= 0

ordem = (pd.Series(bruta['preco_venda'].to_numpy()[validos], index=segmentos[validos])
         .groupby(level=0).median().sort_values(ascending=False).index)            # do mais caro ao mais barato
nome = {s: f'Segmento {chr(65 + i)}' for i, s in enumerate(ordem)} | {-1: 'Sem segmento'}  # mesmos nomes do 05
centroides = X_base[validos].groupby(segmentos[validos]).mean()                    # centro de cada segmento, para o caso sem predict

perfil = (
    bruta.assign(segmento=pd.Series(segmentos).map(nome))
    .groupby('segmento')
    .agg(imoveis=('preco_venda', 'size'), preco_mediano=('preco_venda', 'median'),
         area_habitavel=('area_habitavel', 'median'), idade_imovel=('idade_imovel', 'median'))
)
if classifica_novos:                                                                 # o modelo carregado reproduz a carteira?
    print(f"Concordância do modelo com segmentos.csv: {(modelo.predict(X_base.to_numpy()) == segmentos).mean():.1%}")
perfil.style.format({'preco_mediano': '${:,.0f}', 'area_habitavel': '{:,.0f}'})

Concordância do modelo com segmentos.csv: 100.0%


,imoveis,preco_mediano,area_habitavel,idade_imovel
segmento,,,,
Segmento A,1092,"$222,500","1,668",5.000000
Segmento B,1078,"$144,000","1,208",44.000000
Segmento C,187,"$136,000","1,320",33.000000
Segmento D,570,"$123,500","1,348",83.000000


## 3 · Função de atribuição

Cada imóvel é um dicionário com as colunas da base curada. **Só é preciso informar o que se sabe.** O resto vem do **imóvel mais parecido da carteira**: o que tem as mesmas categorias informadas e os valores numéricos mais próximos.

Por que não usar o valor mais comum de cada coluna? Porque o bairro e o zoneamento mais comuns são os mesmos para qualquer imóvel. Como o one-hot pesa muito na distância, casas opostas acabariam no mesmo segmento.

Colunas ou categorias que não existem na base geram erro com as opções válidas, em vez de um segmento errado silencioso.

In [6]:
numerica = {c: pd.api.types.is_numeric_dtype(bruta_modelo[c]) for c in bruta_modelo.columns}
amplitude = (bruta_modelo.max(numeric_only=True) - bruta_modelo.min(numeric_only=True)).replace(0, 1)  # para comparar colunas em escalas diferentes


def mais_parecido(imovel):
    # Imóvel da carteira com as mesmas categorias informadas e os números mais próximos.
    candidatos = bruta_modelo
    for col, valor in imovel.items():
        if not numerica[col]:
            candidatos = candidatos[candidatos[col] == valor]                     # mesma categoria
    if candidatos.empty:
        raise ValueError(f'Nenhum imóvel da carteira combina estas categorias: {imovel}')
    informadas = [c for c in imovel if numerica[c]]
    if not informadas:
        return candidatos.iloc[0]
    distancia = ((candidatos[informadas] - pd.Series(imovel)[informadas].astype(float)).abs() / amplitude[informadas]).sum(axis=1)
    return candidatos.loc[distancia.idxmin()]                                      # menor diferença somada


def segmentar(imoveis):
    desconhecidas = set().union(*imoveis) - set(bruta_modelo.columns)
    if desconhecidas:
        raise ValueError(f'Colunas que não existem na base curada: {sorted(desconhecidas)}')
    for imovel in imoveis:                                                          # categoria desconhecida viraria zeros no one-hot
        for col, valor in imovel.items():
            if not numerica[col] and valor not in set(bruta_modelo[col]):
                raise ValueError(f'{col}: "{valor}" não existe na base. Válidos: {sorted(map(str, bruta_modelo[col].unique()))}')

    entrada = pd.DataFrame([{**mais_parecido(imovel), **imovel} for imovel in imoveis])[bruta_modelo.columns]  # completa pelo mais parecido
    entrada = entrada.astype(bruta_modelo.dtypes.to_dict())                                                     # mesmos tipos da base

    X_novo = preprocessador.transform(codificar(entrada)).to_numpy()                             # mesmo espaço do modelo
    if classifica_novos:
        rotulos = modelo.predict(X_novo)                                                          # o próprio modelo decide
    else:
        rotulos = centroides.index[cdist(X_novo, centroides).argmin(axis=1)]                     # segmento de centro mais próximo
    return entrada.assign(segmento=[nome[r] for r in rotulos])

## 4 · Entrada: novos imóveis

Edite a lista abaixo e rode as duas células seguintes. Para um lote, use `segmentar(pd.read_csv('arquivo.csv', sep=';').to_dict('records'))`.

As colunas derivadas (`idade_imovel`, `banheiros_total`, `tem_*`…) devem ser coerentes com as originais. Por exemplo, `tem_lareira = 1` junto com o número de lareiras.

In [7]:
novos_imoveis = [
    {'area_habitavel': 2400, 'qualidade_geral': 8, 'idade_imovel': 3, 'anos_desde_reforma': 3,      # casa nova, grande, alto padrão
     'banheiros_total': 3.5, 'tem_lareira': 1, 'tem_segundo_pavimento': 1},
    {'area_habitavel': 950, 'qualidade_geral': 4, 'idade_imovel': 80, 'anos_desde_reforma': 50,     # casa antiga e pequena
     'banheiros_total': 1.0, 'tem_lareira': 0, 'tem_segundo_pavimento': 0},
]

## 5 · Saída: segmento de cada imóvel

In [8]:
resultado = segmentar(novos_imoveis)
informadas = sorted(set().union(*novos_imoveis), key=list(bruta_modelo.columns).index)    # só as colunas que foram digitadas
display(resultado[informadas + ['segmento']])

for i, linha in resultado.iterrows():                                                      # perfil do segmento atribuído
    p = perfil.loc[linha['segmento']]
    display(Markdown(
        f"**Imóvel {i + 1} → {linha['segmento']}** · {p['imoveis']:.0f} imóveis na carteira · "
        f"preço mediano **${p['preco_mediano']:,.0f}** · área mediana {p['area_habitavel']:,.0f} pés² · "
        f"idade mediana {p['idade_imovel']:.0f} anos"
    ))

,qualidade_geral,area_habitavel,idade_imovel,anos_desde_reforma,banheiros_total,tem_lareira,tem_segundo_pavimento,segmento
0,8,2400,3,3,3.5,1,1,Segmento A
1,4,950,80,50,1.0,0,0,Segmento D


**Imóvel 1 → Segmento A** · 1092 imóveis na carteira · preço mediano **$222,500** · área mediana 1,668 pés² · idade mediana 5 anos

**Imóvel 2 → Segmento D** · 570 imóveis na carteira · preço mediano **$123,500** · área mediana 1,348 pés² · idade mediana 83 anos